# Détection de Port de Masque de Sécurité
## Deep Learning avec Transfer Learning (MobileNetV2)

**Objectif:** Classifier des visages en 3 catégories:
- `with_mask` — masque correctement porté
- `without_mask` — sans masque
- `mask_incorrect` — masque mal porté

**Architecture:** Transfer Learning avec MobileNetV2 pré-entraîné sur ImageNet

**Pipeline:**
1. Préparation et exploration des données
2. Augmentation des données
3. Entraînement en deux phases (feature extraction → fine-tuning)
4. Évaluation complète (accuracy, F1, matrice de confusion, ROC)
5. Détection en temps réel


## 0. Installation des dépendances

In [ ]:
# !pip install -r requirements.txt

## 1. Imports et Configuration

In [ ]:
import os
import sys
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from PIL import Image
from tqdm.notebook import tqdm

# Importer notre configuration
import config

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CLASSES = config.CLASSES
NUM_CLASSES = config.NUM_CLASSES
print(f'Classes ({NUM_CLASSES}): {CLASSES}')

## 2. Préparation des Données

In [ ]:
# Créer le dataset de démonstration (ou utilisez votre propre dataset)
# Remplacez par: python prepare_data.py --source /chemin/dataset  pour un vrai dataset

import subprocess
result = subprocess.run([sys.executable, 'prepare_data.py', '--demo'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 3. Exploration du Dataset

In [ ]:
# Compter les images par classe et par split
splits = ['train', 'val', 'test']
counts = {}

for split in splits:
    counts[split] = {}
    for cls in CLASSES:
        path = os.path.join(config.DATA_DIR, split, cls)
        if os.path.isdir(path):
            n = len([f for f in os.listdir(path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            counts[split][cls] = n
        else:
            counts[split][cls] = 0

# Affichage
print('=== Distribution du Dataset ===')
print(f'{"":30s}', end='')
for split in splits:
    print(f'{split:>12s}', end='')
print()
for cls in CLASSES:
    print(f'{cls:30s}', end='')
    for split in splits:
        print(f'{counts[split][cls]:>12d}', end='')
    print()
print(f'{"TOTAL":30s}', end='')
for split in splits:
    print(f'{sum(counts[split].values()):>12d}', end='')
print()

In [ ]:
# Visualisation de la distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#4CAF50', '#F44336', '#FF9800']

for ax, split in zip(axes, splits):
    vals = [counts[split][cls] for cls in CLASSES]
    bars = ax.bar(CLASSES, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Split: {split}  (total={sum(vals)})', fontweight='bold')
    ax.set_ylabel('Nombre d\'images')
    ax.set_xticklabels(CLASSES, rotation=15, ha='right')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontsize=10)

plt.suptitle('Distribution du Dataset par Split et par Classe', y=1.02, fontsize=13)
plt.tight_layout()
os.makedirs(config.RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(config.RESULTS_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualiser des exemples d'images
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
colors_map = {'with_mask': '#4CAF50', 'without_mask': '#F44336', 'mask_incorrect': '#FF9800'}

for row, cls in enumerate(CLASSES):
    cls_path = os.path.join(config.TRAIN_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    images = sorted([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:6]
    for col, img_name in enumerate(images):
        img = Image.open(os.path.join(cls_path, img_name)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, color=colors_map[cls], fontweight='bold')

plt.suptitle('Exemples d\'images par classe', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Prétraitement et Augmentation des Données

In [ ]:
from dataset import get_transforms, build_dataloaders

loaders, classes = build_dataloaders()
train_loader = loaders['train']
val_loader   = loaders['val']
test_loader  = loaders['test']

In [ ]:
# Visualiser les augmentations appliquées
img_path = None
for cls in CLASSES:
    p = os.path.join(config.TRAIN_DIR, cls)
    if os.path.isdir(p):
        files = os.listdir(p)
        if files:
            img_path = os.path.join(p, files[0])
            break

if img_path:
    original = Image.open(img_path).convert('RGB')
    train_transform = get_transforms('train')
    mean = torch.tensor(config.MEAN).view(3,1,1)
    std  = torch.tensor(config.STD).view(3,1,1)

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes[0, 0].imshow(original)
    axes[0, 0].set_title('Originale', fontweight='bold')
    axes[0, 0].axis('off')

    for i in range(1, 5):
        t = train_transform(original)
        t = (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[0, i].imshow(t)
        axes[0, i].set_title(f'Augmentée #{i}')
        axes[0, i].axis('off')

    val_transform = get_transforms('val')
    t = val_transform(original)
    t = (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[1, 0].imshow(t)
    axes[1, 0].set_title('Val/Test (resize)', fontweight='bold')
    axes[1, 0].axis('off')
    for i in range(1, 5):
        axes[1, i].axis('off')

    plt.suptitle('Exemples d\'augmentation de données', fontsize=12)
    plt.tight_layout()
    plt.show()

## 5. Architecture du Modèle (Transfer Learning)

In [ ]:
from model import build_model, freeze_backbone, unfreeze_all

model = build_model(num_classes=NUM_CLASSES, model_name='mobilenet_v2')
model = model.to(device)

print('Architecture MobileNetV2 modifiée:')
print('=' * 50)
total_params = sum(p.numel() for p in model.parameters())
print(f'Paramètres totaux:        {total_params:>12,}')

freeze_backbone(model, 'mobilenet_v2')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres entraînables:  {trainable:>12,}  ({100*trainable/total_params:.1f}%)')
print()
print('Tête de classification personnalisée:')
print(model.classifier)

## 6. Entraînement

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc='Train', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc='Val  ', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

In [ ]:
# Hyperparamètres
NUM_EPOCHS  = 30
LR_PHASE1   = 1e-3
LR_PHASE2   = 1e-4
PHASE1_END  = 10    # époques avec backbone gelé
PATIENCE    = 7

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1, weight_decay=1e-4
)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5, verbose=True)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
patience_ctr = 0
os.makedirs(config.CHECKPOINTS_DIR, exist_ok=True)
best_path = os.path.join(config.CHECKPOINTS_DIR, 'best_model.pth')

print('Début de l\'entraînement...')
print(f'Phase 1 (backbone gelé): époques 1-{PHASE1_END}')
print(f'Phase 2 (fine-tuning):   époques {PHASE1_END+1}-{NUM_EPOCHS}')

for epoch in range(1, NUM_EPOCHS + 1):
    # Transition vers Phase 2
    if epoch == PHASE1_END + 1:
        print(f'\n>>> Phase 2: Fine-tuning de tout le réseau (epoch {epoch}) <<<')
        unfreeze_all(model)
        optimizer = optim.Adam(model.parameters(), lr=LR_PHASE2, weight_decay=1e-4)
        scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5, verbose=True)

    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = eval_epoch(model, val_loader, criterion, device)
    scheduler.step(val_acc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train Loss={train_loss:.4f} Acc={train_acc:.4f} | '
          f'Val Loss={val_loss:.4f} Acc={val_acc:.4f} | '
          f'LR={optimizer.param_groups[0]["lr"]:.2e} | '
          f'{time.time()-t0:.1f}s')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_acc': val_acc}, best_path)
        print(f'  ✓ Meilleur modèle sauvegardé (val_acc={val_acc:.4f})')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping à l\'époque {epoch}.')
            break

print(f'\nEntraînement terminé. Meilleure val_acc: {best_val_acc:.4f}')

In [ ]:
# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], 'b-o', label='Train', markersize=3)
axes[0].plot(history['val_loss'],   'r-o', label='Validation', markersize=3)
if PHASE1_END < len(history['train_loss']):
    axes[0].axvline(x=PHASE1_END-1, color='gray', linestyle='--', label='Début Fine-tuning')
axes[0].set_title('Loss par Époque', fontweight='bold')
axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], 'b-o', label='Train', markersize=3)
axes[1].plot(history['val_acc'],   'r-o', label='Validation', markersize=3)
if PHASE1_END < len(history['val_acc']):
    axes[1].axvline(x=PHASE1_END-1, color='gray', linestyle='--', label='Début Fine-tuning')
axes[1].set_title('Accuracy par Époque', fontweight='bold')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Courbes d\'entraînement', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 7. Évaluation Complète

In [ ]:
# Charger le meilleur modèle
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Meilleur modèle chargé (epoch={checkpoint['epoch']}, val_acc={checkpoint['val_acc']:.4f})")

# Prédictions sur le test set
@torch.no_grad()
def get_predictions(loader):
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in tqdm(loader, desc='Prédiction'):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.append(probs)
    return np.array(all_labels), np.array(all_preds), np.vstack(all_probs)

y_true, y_pred, y_probs = get_predictions(test_loader)
acc = (y_true == y_pred).mean()
print(f'\nAccuracy sur le test set: {acc:.4f} ({acc*100:.2f}%)')

In [ ]:
# Rapport de classification
print('\n=== Rapport de Classification ===')
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2%'],
    ['Matrice de confusion (nombres)', 'Matrice de confusion (normalisée)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax,
                linewidths=0.5, linecolor='gray')
    ax.set_xlabel('Prédiction', fontweight='bold')
    ax.set_ylabel('Vérité terrain', fontweight='bold')
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# Courbes ROC
y_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
colors_roc = ['#2196F3', '#F44336', '#FF9800']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC={roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('Taux de faux positifs'); axes[0].set_ylabel('Taux de vrais positifs')
axes[0].set_title('Courbes ROC', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Distribution des probabilités
for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    mask_correct = (y_true == i)
    axes[1].hist(y_probs[mask_correct, i], bins=20, alpha=0.6,
                 color=color, label=f'{cls} (n={mask_correct.sum()})', density=True)
axes[1].set_xlabel('Probabilité prédite'); axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution des probabilités (vraie classe)', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'roc_and_proba.png'), dpi=150)
plt.show()

In [ ]:
# Exemples de prédictions correctes et incorrectes
test_dataset = datasets.ImageFolder(config.TEST_DIR, transform=get_transforms('val'))

correct_indices   = np.where(y_true == y_pred)[0]
incorrect_indices = np.where(y_true != y_pred)[0]

mean_t = torch.tensor(config.MEAN).view(3,1,1)
std_t  = torch.tensor(config.STD).view(3,1,1)

def show_examples(indices, title, n=6):
    if len(indices) == 0:
        print(f'Aucun exemple pour: {title}')
        return
    chosen = np.random.choice(indices, min(n, len(indices)), replace=False)
    fig, axes = plt.subplots(1, len(chosen), figsize=(3*len(chosen), 4))
    if len(chosen) == 1:
        axes = [axes]
    for ax, idx in zip(axes, chosen):
        img_t, _ = test_dataset[idx]
        img = (img_t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(img)
        pred_cls  = classes[y_pred[idx]]
        true_cls  = classes[y_true[idx]]
        conf      = y_probs[idx, y_pred[idx]]
        color     = 'green' if y_true[idx] == y_pred[idx] else 'red'
        ax.set_title(f'Pred: {pred_cls}\nVrai: {true_cls}\nConf: {conf:.2%}',
                     color=color, fontsize=8)
        ax.axis('off')
    plt.suptitle(title, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_examples(correct_indices,   'Prédictions Correctes ✓', n=6)
show_examples(incorrect_indices, 'Prédictions Incorrectes ✗', n=6)

## 8. Récapitulatif des Résultats

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print('=' * 60)
print('       RÉCAPITULATIF DES PERFORMANCES DU MODÈLE')
print('=' * 60)
print(f'Modèle:               MobileNetV2 (Transfer Learning)')
print(f'Dataset:              {sum(counts["train"].values())} train | {sum(counts["val"].values())} val | {sum(counts["test"].values())} test')
print(f'Époques entraînées:   {len(history["train_acc"])}')
print(f'Meilleure Val Acc:    {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'Test Accuracy:        {acc:.4f} ({acc*100:.2f}%)')
print(f'Test Precision (macro): {precision_score(y_true, y_pred, average="macro"):.4f}')
print(f'Test Recall (macro):    {recall_score(y_true, y_pred, average="macro"):.4f}')
print(f'Test F1 (macro):        {f1_score(y_true, y_pred, average="macro"):.4f}')
print('=' * 60)
print(f'Résultats sauvegardés dans: {config.RESULTS_DIR}/')
print(f'Modèle sauvegardé dans:     {best_path}')

## 9. Inférence sur une image unique

In [ ]:
def predict_image(image_path, model, classes, device, show=True):
    """Prédit la classe d'une image de visage."""
    transform = get_transforms('val')
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
    
    pred_idx = np.argmax(probs)
    pred_cls = classes[pred_idx]
    confidence = probs[pred_idx]
    
    if show:
        colors_bar = ['#4CAF50', '#F44336', '#FF9800']
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(img)
        axes[0].set_title(f'{config.LABELS_FR.get(pred_cls, pred_cls)}\n({confidence:.2%})',
                          fontweight='bold', fontsize=11)
        axes[0].axis('off')
        
        bars = axes[1].barh(classes, probs, color=colors_bar[:len(classes)])
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('Probabilité')
        axes[1].set_title('Distribution des probabilités')
        for bar, prob in zip(bars, probs):
            axes[1].text(prob + 0.01, bar.get_y() + bar.get_height()/2,
                         f'{prob:.3f}', va='center')
        plt.tight_layout()
        plt.show()
    
    return pred_cls, confidence, probs

# Tester sur une image du test set
test_cls = CLASSES[0]
test_cls_path = os.path.join(config.TEST_DIR, test_cls)
if os.path.isdir(test_cls_path):
    test_imgs = os.listdir(test_cls_path)
    if test_imgs:
        test_img_path = os.path.join(test_cls_path, test_imgs[0])
        cls, conf, probs = predict_image(test_img_path, model, classes, device)
        print(f'Prédiction: {cls} ({conf:.2%})')

## 10. Détection en Temps Réel

Pour la détection en temps réel via webcam, exécutez dans un terminal:
```bash
# Webcam
python detect.py --source 0

# Sur une image
python detect.py --source chemin/vers/image.jpg

# Sur une vidéo
python detect.py --source chemin/vers/video.mp4
```

Le script:
1. Détecte les visages avec OpenCV DNN (SSD ResNet10)
2. Classifie chaque visage avec notre modèle entraîné
3. Affiche le résultat avec un rectangle coloré:
   - 🟢 Vert = masque correctement porté
   - 🔴 Rouge = sans masque
   - 🟠 Orange = masque mal porté